In [10]:
import pandas as pd
import requests
from io import BytesIO
from PIL import Image
import os

print("Setup done")

Setup done


In [11]:
df = pd.read_csv('AI_Classification-Project.csv', keep_default_na=False)
print(df.shape)
print(df['label'].value_counts())
df.head()

(6557, 17)
label
FAKE    3767
REAL    2790
Name: count, dtype: int64


,image_id,image_url,label,label_numeric,category,gender,age_group,source,fake_method,image_quality,resolution,confidence_score,detection_difficulty,dataset_split,date_collected,version,year
0,1,https://images.unsplash.com/photo-172422561835...,REAL,1,Authentic,Female,18-25,Unsplash,None,High,1080x1080,0.96,Easy,train,24-04-2026,v2.0,2026
1,2,https://images.unsplash.com/photo-172422561873...,REAL,1,Authentic,Male,50+,Unsplash,None,High,1080x1080,0.97,Easy,train,04-04-2026,v2.0,2026
2,3,https://images.unsplash.com/photo-172422561834...,REAL,1,Authentic,Female,36-50,Unsplash,None,High,1080x1080,0.95,Easy,test,07-02-2026,v2.0,2026
3,4,https://images.unsplash.com/photo-172422561855...,REAL,1,Authentic,Female,26-35,Unsplash,None,High,1080x1080,0.89,Easy,train,20-02-2026,v2.0,2026
4,5,https://images.unsplash.com/photo-172422561823...,REAL,1,Authentic,Male,36-50,Unsplash,None,High,1080x1080,0.86,Easy,train,12-04-2026,v2.0,2026


In [12]:
real_subset = df[df['label'] == 'REAL'].sample(1000, random_state=42)
fake_subset = df[df['label'] == 'FAKE'].sample(1000, random_state=42)
subset = pd.concat([real_subset, fake_subset]).reset_index(drop=True)
print(subset.shape)

(2000, 17)


In [13]:
os.makedirs('images', exist_ok=True)
failed = []

for i, row in subset.iterrows():
    try:
        resp = requests.get(row['image_url'], timeout=10)
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        img = img.resize((224, 224))
        img.save(f"images/{row['image_id']}_{row['label']}.jpg")
    except Exception as e:
        failed.append(row['image_id'])
    if i % 100 == 0:
        print(f"Processed {i}/{len(subset)}")

print(f"Done. Failed: {len(failed)}")

Processed 0/2000
Processed 100/2000
Processed 200/2000
Processed 300/2000
Processed 400/2000
Processed 500/2000
Processed 600/2000
Processed 700/2000
Processed 800/2000
Processed 900/2000
Processed 1000/2000
Processed 1100/2000
Processed 1200/2000
Processed 1300/2000
Processed 1400/2000
Processed 1500/2000
Processed 1600/2000
Processed 1700/2000
Processed 1800/2000
Processed 1900/2000
Done. Failed: 242


In [14]:
print(len(os.listdir('images')), "images saved")

1758 images saved


In [15]:
import torch
import torchvision
from torchvision import transforms, models
import numpy as np
from sklearn.model_selection import train_test_split

print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Torch version: 2.11.0+cu130
GPU available: True


In [16]:
image_files = os.listdir('images')
print(f"Total images: {len(image_files)}")

paths = []
labels = []
for f in image_files:
    paths.append(f"images/{f}")
    labels.append(1 if "_REAL" in f else 0)   # 1 = REAL, 0 = FAKE

print("REAL:", sum(labels), "FAKE:", len(labels) - sum(labels))

Total images: 1758
REAL: 1000 FAKE: 758


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = torch.nn.Identity()
model = model.to(device)
model.eval()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Model loaded")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 156MB/s]


Model loaded


In [18]:
features = []
valid_labels = []

with torch.no_grad():
    for i, p in enumerate(paths):
        try:
            img = Image.open(p).convert('RGB')
            tensor = preprocess(img).unsqueeze(0).to(device)
            feat = model(tensor).cpu().numpy().flatten()
            features.append(feat)
            valid_labels.append(labels[i])
        except Exception as e:
            print(f"Skipped {p}: {e}")
        if i % 200 == 0:
            print(f"Processed {i}/{len(paths)}")

features = np.array(features)
valid_labels = np.array(valid_labels)
print("Feature matrix shape:", features.shape)

Processed 0/1758
Processed 200/1758
Processed 400/1758
Processed 600/1758
Processed 800/1758
Processed 1000/1758
Processed 1200/1758
Processed 1400/1758
Processed 1600/1758
Feature matrix shape: (1758, 2048)


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features, valid_labels, test_size=0.2, random_state=42, stratify=valid_labels
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (1406, 2048) Test: (352, 2048)


In [20]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
print("Training done")

Training done


In [21]:
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
print("Predictions done")

Predictions done


In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")

Accuracy:  0.9744
Precision: 0.9897
Recall:    0.9650
F1-Score:  0.9772


In [23]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix (rows=actual, cols=predicted):")
print("        FAKE  REAL")
print(f"FAKE    {cm[0][0]:4d}  {cm[0][1]:4d}")
print(f"REAL    {cm[1][0]:4d}  {cm[1][1]:4d}")

Confusion Matrix (rows=actual, cols=predicted):
        FAKE  REAL
FAKE     150     2
REAL       7   193


In [24]:
import joblib
joblib.dump(clf, 'face_classifier.pkl')
print("Model saved")

Model saved


In [25]:
#re-run predictions on ALL data to get full breakdown by demographics
all_pred = clf.predict(features)
all_proba = clf.predict_proba(features)[:, 1]

# Build a results dataframe with image_id and metadata
image_ids = [int(p.split('/')[-1].split('_')[0]) for p in paths]
results_df = pd.DataFrame({
    'image_id': image_ids,
    'actual': valid_labels,
    'predicted': all_pred
})
results_df = results_df.merge(subset[['image_id','gender','age_group','image_quality']], on='image_id')
results_df['correct'] = results_df['actual'] == results_df['predicted']

print("Accuracy by gender:")
print(results_df.groupby('gender')['correct'].mean())
print("\nAccuracy by age group:")
print(results_df.groupby('age_group')['correct'].mean())
print("\nAccuracy by image quality:")
print(results_df.groupby('image_quality')['correct'].mean())

Accuracy by gender:
gender
Female     0.997199
Male       0.988827
Unknown    0.996165
Name: correct, dtype: float64

Accuracy by age group:
age_group
18-25    0.993151
26-35    0.995475
36-50    0.993377
50+      0.997647
Name: correct, dtype: float64

Accuracy by image quality:
image_quality
High      0.994945
Medium    0.994746
Name: correct, dtype: float64


In [26]:
!pip install gradio -q

In [ ]:
import gradio as gr
import numpy as np

def predict_face(image):
    if isinstance(image, np.ndarray):
        img = Image.fromarray(image).convert('RGB')
    else:
        img = image.convert('RGB')
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model(tensor).cpu().numpy()
    pred = clf.predict(feat)[0]
    proba = clf.predict_proba(feat)[0]
    label = "REAL" if pred == 1 else "FAKE"
    confidence = proba[pred] * 100
    return f"{label} ({confidence:.2f}% confidence)"

demo = gr.Interface(
    fn=predict_face,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="AI Face Authenticity Classifier",
    description="Upload a face image to check if it's REAL or AI-Generated (FAKE)"
)

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fca82e9f296fb852c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
